In [13]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from duckduckgo_search import DDGS

from typing import List, Dict
from ddgs import DDGS

load_dotenv()

True

In [6]:
from typing import TypedDict
class State(TypedDict):
    messages: Annotated[list,add_messages]  

In [7]:
llm = ChatGroq(model_name="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY") )

## TOOLS

In [15]:
@tool
def duckduckgo_search_tool(query: str, max_results: int = 5) -> str:
    """Search the web via DuckDuckGo for fresh/factual info. Returns formatted results."""
    try:
        with DDGS() as ddg:
            results = list(ddg.text(query, max_results=max_results))
    except Exception as e:
        return f"Search failed: {e}"

    if not results:
        return f"No results found for: {query}"

    return "\n\n".join(
        f"{r.get('title','')}\n{r.get('href','')}\n{r.get('body','')}"
        for r in results
    )



In [16]:
tools = [duckduckgo_search_tool]
llm_with_tools = llm.bind_tools(tools)

In [17]:
from langgraph.prebuilt import ToolNode, tools_condition

def tool_calling_node(state: State):
    system_prompt = """You are a helpful assistant with one tool: duckduckgo_search_tool(query, max_results).

Use it ONLY when you need fresh, real-world, or factual info you don't already know (news, scores, prices, current events, recent facts). For chit-chat, opinions, or things you already know, answer directly without calling it.

When you call the tool, summarise the returned results clearly and cite nothing you didn't get back. Do not invent facts. Do not call any tool other than duckduckgo_search_tool."""

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(State)
builder.add_node("tool_calling_node", tool_calling_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_node")
builder.add_conditional_edges("tool_calling_node", tools_condition)
builder.add_edge("tools", "tool_calling_node")

graph = builder.compile()

In [18]:
from groq import BadRequestError

messages = []  # Persistent conversation history

while 1:
    input_text = input()
    if input_text.lower() == "exit":
        break
    else: 
        messages.append(HumanMessage(content=input_text))  # Add user message
        try:
            result2 = graph.invoke({"messages": messages})
            response = result2['messages'][-1]
            response.pretty_print()
            messages.append(response)  # Add assistant response
        except BadRequestError as e:
            error_msg = str(e)
            print(f"Error: The assistant tried to use a capability that isn't available. Please rephrase your request.")
            print(f"Technical details: {error_msg}")
            # Remove the last user message that caused the error
            messages.pop()

================================== Ai Message ==================================

Hello! How can I help you today?
================================== Ai Message ==================================

My training data includes information up to **June 2024**. If you need details about events or developments after that date, let me know and I can look up the latest information for you.
================================== Ai Message ==================================

**Gold spot price (XAU / USD)**  
- **Current price:** **≈ $4,450.40 per troy ounce** (Kitco live spot price, bid price)  

*Note:* Gold prices change every few seconds during market hours. The figure above reflects the latest bid price reported by Kitco at the time of the search. If you need a continuously updated quote, you can check a live‑price site such as [Kitco](https://www.kitco.com/charts/gold) or a financial platform.
